In [20]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor

In [29]:
df = pd.read_csv("ml_dataset.csv")
df["delivery_day"] = pd.to_datetime(df["delivery_day"])

print(df.shape)
print(df.head())

(224307, 54)
  delivery_day  hour  price_value  buy_volume_value  sell_volume_value  \
0   2015-01-01     0        27.40            4849.8             4758.9   
1   2015-01-01     1        27.07            4786.3             4431.1   
2   2015-01-01     2        26.93            4719.0             4328.0   
3   2015-01-01     3        26.83            4692.2             4275.4   
4   2015-01-01     4        26.81            4682.4             4264.5   

   net_volume  flow_DK1_export  flow_DK1_import  flow_GER_export  \
0        90.9              0.0           1131.3              0.0   
1       355.2              0.0           1532.0              0.0   
2       391.0              0.0           1532.0              0.0   
3       416.8              0.0           1532.0              0.0   
4       417.9              0.0           1532.0              0.0   

   flow_GER_import  ...  day_of_year  price_lag_24  price_lag_168  \
0              0.0  ...            1           NaN            Na

In [30]:
print(df.columns)

Index(['delivery_day', 'hour', 'price_value', 'buy_volume_value',
       'sell_volume_value', 'net_volume', 'flow_DK1_export', 'flow_DK1_import',
       'flow_GER_export', 'flow_GER_import', 'flow_NL_export',
       'flow_NL_import', 'flow_NO1_export', 'flow_NO1_import',
       'flow_NO5_export', 'flow_NO5_import', 'capacity_GER', 'capacity_NL',
       'capacity_NO1', 'capacity_NO5', 'util_GER', 'util_NL', 'util_NO1',
       'util_NO5', 'capacity_missing_DK1', 'capacity_missing_GER',
       'capacity_missing_NL', 'capacity_missing_NO1', 'capacity_missing_NO5',
       'capacity_zero_DK1', 'capacity_zero_GER', 'capacity_zero_NL',
       'capacity_zero_NO1', 'capacity_zero_NO5', 'price_DK1', 'price_NO1',
       'price_NO5', 'total_exports', 'total_imports', 'net_flow',
       'day_of_week', 'hour_sin', 'hour_cos', 'month', 'day_of_year',
       'price_lag_24', 'price_lag_168', 'price_value_lag24', 'price_DK1_lag24',
       'price_NO1_lag24', 'price_NO5_lag24', 'generation_forecast',
     

In [31]:
# Target: price 24 hours ahead
df = df.sort_values(["delivery_day", "hour"]).reset_index(drop=True)

df["target_price_next_day"] = df["price_value"].shift(-24)

# Drop rows without a future price (last 24 rows)
df = df.dropna(subset=["target_price_next_day"])

print(df.shape)
df[["delivery_day", "hour", "price_value", "target_price_next_day"]].head(10)

(224283, 55)


,delivery_day,hour,price_value,target_price_next_day
0,2015-01-01,0,27.40,26.71
1,2015-01-01,1,27.07,26.41
2,2015-01-01,2,26.93,26.20
3,2015-01-01,3,26.83,25.60
4,2015-01-01,4,26.81,24.74
5,2015-01-01,5,26.97,26.10
6,2015-01-01,6,27.07,27.25
7,2015-01-01,7,26.80,28.37
8,2015-01-01,8,26.97,28.37
9,2015-01-01,9,26.98,28.32


In [32]:
drop_cols = [
    "delivery_day",
]

X = df.drop(columns=drop_cols + ["target_price_next_day"])
y = df["target_price_next_day"]

print("Features:", len(X.columns))

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows : {len(X_test):,}")

Features: 53
Train rows: 179,426
Test rows : 44,857


In [33]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
print("done")

Training...


KeyboardInterrupt: 

In [15]:
rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

rmse = root_mean_squared_error(
y_test,
X_test["price_value_lag24"]
)
print(f"price_value_lag24 RMSE: {rmse:.4f}")

RMSE = 25.7843
R²: 0.6138610003815266
price_value_lag24 RMSE: 36.1600


In [51]:
model = XGBRegressor(
        n_estimators=500,
        max_depth=3,
        min_child_weight=5,
        learning_rate=0.01,
        objective="reg:squarederror"
    )

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
print("done")

Training...
Predicting...
done


In [52]:
rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

rmse = root_mean_squared_error(
y_test,
X_test["price_value_lag24"]
)
print(f"price_value_lag24 RMSE: {rmse:.4f}")

RMSE = 25.2413
R²: 0.6668508661670822
price_value_lag24 RMSE: 37.1688


In [36]:
importance = (
pd.DataFrame({
"feature": X.columns,
"importance": model.feature_importances_
})
.sort_values("importance", ascending=False)
)
print(importance.head(20))

                feature  importance
1           price_value    0.528817
35            price_NO5    0.050464
33            price_DK1    0.043197
18         capacity_NO5    0.040342
44         price_lag_24    0.039897
45        price_lag_168    0.038782
42                month    0.024205
52        wind_forecast    0.012857
3     sell_volume_value    0.012625
39          day_of_week    0.011529
43          day_of_year    0.011431
4            net_volume    0.011357
14      flow_NO5_import    0.010214
50  generation_forecast    0.010143
12      flow_NO1_import    0.009006
48      price_NO1_lag24    0.008891
49      price_NO5_lag24    0.008667
2      buy_volume_value    0.008357
38             net_flow    0.008173
51        load_forecast    0.008144
